In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler

In [2]:
df_normal = pd.read_csv(
    r"C:\Users\Arushi Sharma\Downloads\Master Thesis\normal_2\normal_2\conn.csv",
    header=None
)

df_ddos = pd.read_csv(
    r"C:\Users\Arushi Sharma\Downloads\Master Thesis\normal_DDoS_1\normal_DDoS_1\conn.csv",
    header=None
)

In [3]:
print(df_normal.head())

           0                   1              2      3              4     5   \
0  1554230127   Cwz6DH3Vc1wHxazgU      127.0.0.1  42100      127.0.0.1  7878   
1  1554230127  Cua57j2bGKNBfAVH56  192.168.1.190  43530  192.168.1.190  7878   
2  1554230133   C1bNNk5ApS9xoKfFh  192.168.1.195  49812  117.18.237.29    80   
3  1554230133  CNuAH92t40io1fQ8E2      127.0.0.1  42100      127.0.0.1  7878   
4  1554230133   Cgsjb3PKvE9xqXMk5  192.168.1.190  43530  192.168.1.190  7878   

    6  7         8  9   ...   11 12 13 14   15 16   17  18  19  20  
0  tcp  -         -  -  ...  OTH  -  -  0  CcC  0    0   0   0   -  
1  tcp  -         -  -  ...  OTH  -  -  0  CcC  0    0   0   0   -  
2  tcp  -  4.523548  0  ...   SH  -  -  0    F  5  200   0   0   -  
3  tcp  -         -  -  ...  OTH  -  -  0  CcC  0    0   0   0   -  
4  tcp  -         -  -  ...  OTH  -  -  0  CcC  0    0   0   0   -  

[5 rows x 21 columns]


In [4]:
columns = [
    "ts", "uid", "id.orig_h", "id.orig_p", "id.resp_h", "id.resp_p",
    "proto", "service", "duration", "orig_bytes", "resp_bytes",
    "conn_state", "local_orig", "local_resp", "missed_bytes",
    "history", "orig_pkts", "orig_ip_bytes", "resp_pkts",
    "resp_ip_bytes", "tunnel_parents"
]
df_normal.columns = columns
df_ddos.columns   = columns

In [5]:
print(df_normal.columns)

Index(['ts', 'uid', 'id.orig_h', 'id.orig_p', 'id.resp_h', 'id.resp_p',
       'proto', 'service', 'duration', 'orig_bytes', 'resp_bytes',
       'conn_state', 'local_orig', 'local_resp', 'missed_bytes', 'history',
       'orig_pkts', 'orig_ip_bytes', 'resp_pkts', 'resp_ip_bytes',
       'tunnel_parents'],
      dtype='object')


In [6]:
assert 'ts' in df_normal.columns, "Timestamp column missing!"

In [7]:
df_combined = pd.concat([df_normal, df_ddos], ignore_index=True)

def preprocess_combined(df):
    df = df.copy()
    
    ts = df['ts']
    
    df = df.drop(columns=[
        'ts',
        'uid',
        'id.orig_h',
        'id.resp_h',
        'tunnel_parents'
    ], errors='ignore')
    
    df = pd.get_dummies(df, columns=[
        'proto',
        'service',
        'conn_state',
        'history'
    ], drop_first=True)
    
    df = df.apply(pd.to_numeric, errors='coerce')
    df = df.fillna(0)
    
    return df, ts

df_all_proc, ts_all = preprocess_combined(df_combined)

In [8]:
df_normal_proc = df_all_proc.iloc[:len(df_normal)].reset_index(drop=True)
df_ddos_proc   = df_all_proc.iloc[len(df_normal):].reset_index(drop=True)

ts_normal = ts_all.iloc[:len(df_normal)].reset_index(drop=True)
ts_ddos   = ts_all.iloc[len(df_normal):].reset_index(drop=True)

In [13]:
scaler = StandardScaler()

scaler.fit(df_normal_proc)

X_normal = scaler.transform(df_normal_proc)
X_ddos   = scaler.transform(df_ddos_proc)

In [14]:
print(df_normal_proc.shape)
print(df_ddos_proc.shape)

print(df_normal_proc.columns.equals(df_ddos_proc.columns))

(7179, 774)
(123536, 774)
True


In [15]:
df_ddos_proc = df_ddos_proc.reindex(columns=df_normal_proc.columns, fill_value=0)

In [16]:
print(df_normal_proc.columns.equals(df_ddos_proc.columns))

True


In [17]:
scaler = StandardScaler()

scaler.fit(df_normal_proc)

X_normal = scaler.transform(df_normal_proc)
X_ddos   = scaler.transform(df_ddos_proc)

In [18]:
print(X_normal.shape)
print(X_ddos.shape)

(7179, 774)
(123536, 774)


In [19]:
assert list(df_normal_proc.columns) == list(df_ddos_proc.columns), "Feature mismatch!"

In [20]:
df_normal_processed = pd.DataFrame(X_normal, columns=df_normal_proc.columns)
df_ddos_processed   = pd.DataFrame(X_ddos, columns=df_ddos_proc.columns)

df_normal_processed['ts'] = ts_normal.values
df_ddos_processed['ts']   = ts_ddos.values

In [21]:
df_normal_processed = pd.DataFrame(X_normal, columns=df_normal_proc.columns)
df_ddos_processed   = pd.DataFrame(X_ddos, columns=df_ddos_proc.columns)

In [22]:
df_normal_processed['ts'] = ts_normal.values
df_ddos_processed['ts']   = ts_ddos.values

In [23]:
print(df_normal_processed.shape)
print(df_ddos_processed.shape)

print(df_normal_processed.head())

(7179, 775)
(123536, 775)
   id.orig_p  id.resp_p  duration  orig_bytes  resp_bytes  local_orig  \
0   0.240637   0.225381 -0.055693    -0.01445   -0.014109         0.0   
1   0.329851   0.225381 -0.055693    -0.01445   -0.014109         0.0   
2   0.721768  -0.726290 -0.042762    -0.01445   -0.014109         0.0   
3   0.240637   0.225381 -0.055693    -0.01445   -0.014109         0.0   
4   0.329851   0.225381 -0.055693    -0.01445   -0.014109         0.0   

   local_resp  missed_bytes  orig_pkts  orig_ip_bytes  ...  history_^hadtAF  \
0         0.0     -0.012241  -0.024979      -0.018184  ...              0.0   
1         0.0     -0.012241  -0.024979      -0.018184  ...              0.0   
2         0.0     -0.012241  -0.018267      -0.017450  ...              0.0   
3         0.0     -0.012241  -0.024979      -0.018184  ...              0.0   
4         0.0     -0.012241  -0.024979      -0.018184  ...              0.0   

   history_^hdA  history_^hdADa  history_^hdAF  history_^hdA

In [24]:
df_normal_processed.to_csv("df_normal_processed.csv", index=False)
df_ddos_processed.to_csv("df_ddos_processed.csv", index=False)

In [27]:
# Save processed datasets for next stage

df_normal_processed.to_csv("df_normal_processed.csv", index=False)
df_ddos_processed.to_csv("df_ddos_processed.csv", index=False)

print("✅ Saved processed datasets")
print("Normal shape:", df_normal_processed.shape)
print("DDoS shape:", df_ddos_processed.shape)

✅ Saved processed datasets
Normal shape: (7179, 775)
DDoS shape: (123536, 775)
